加载网页类型文档

In [1]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

page_url = "https://docs.langchain.com/langsmith/deploy-overview"

loader = WebBaseLoader(web_paths=[page_url])
docs = []
async for doc in loader.alazy_load():
    docs.append(doc)

assert  len(docs) == 1
doc = docs[0]

USER_AGENT environment variable not set, consider setting it to identify your requests.
Fetching pages: 100%|##########| 1/1 [00:02<00:00,  2.66s/it]


In [2]:
print(f"{doc.metadata}\n")
print("全文字符数：", len(doc.page_content))
print(doc.page_content)

{'source': 'https://docs.langchain.com/langsmith/deploy-overview', 'title': 'Deploy agents - Docs by LangChain', 'description': 'Deploy agents to production on LangSmith Agent Server, on LangSmith Cloud, BYOC, your own infrastructure, or a standalone server.', 'language': 'en'}

全文字符数： 5960






Deploy agents - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageDeploySearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDeploy agentsGet startedAgent ServerDeploy to CloudDeploy to Self-hostedPrompt & Context HubSandboxesOverviewQuickstartDeployment componentsOverviewData planeControl planeDevelop & testLocal development & testingStudioFrameworks and platformsDeploy Google ADKClaude SDK, Str

## 按语义切分网页正文
先运行上面的网页加载单元格，再依次运行以下单元格。语义切分通过句子向量距离寻找断点，文本块长度不固定。

In [ ]:
# Install into the current Jupyter kernel using the Tsinghua PyPI mirror.
%pip install -i https://pypi.tuna.tsinghua.edu.cn/simple --timeout 60 langchain-huggingface sentence-transformers


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

# Free local model for the English web page; no API key required.
# The first run downloads the model. Later runs use the local cache.
# For Chinese text: BAAI/bge-small-zh-v1.5
# For multilingual text: BAAI/bge-m3 (requires more memory)
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    buffer_size=1,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95,
    sentence_split_regex=r"(?<=[\u3002\uff01\uff1f])|(?<=[.!?])\s+|\n{2,}",
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

F:\dev\.conda\envs\langchain1.2\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Colin\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (5 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# 使用上面 WebBaseLoader 拉取的 docs，保留 source、title 等元数据。
if "docs" not in globals() or not docs:
    raise ValueError("请先运行上面的网页加载单元格，得到 docs")

web_docs = [doc for doc in docs if doc.page_content.strip()]
if not web_docs:
    raise ValueError("网页没有可切分的正文，请检查加载结果")

# Run embeddings locally on CPU; save chunks for retrieval.
semantic_docs = semantic_splitter.split_documents(web_docs)
for index, chunk in enumerate(semantic_docs, start=1):
    chunk.metadata["chunk_index"] = index

print(f"网页文档数：{len(web_docs)}")
print(f"原文总字符数：{sum(len(doc.page_content) for doc in web_docs)}")
print(f"语义文本块数：{len(semantic_docs)}")

for chunk in semantic_docs[:5]:
    print(f"\n{'=' * 60}")
    print(f"文本块 {chunk.metadata['chunk_index']} | 字符数：{len(chunk.page_content)}")
    print(f"元数据：{chunk.metadata}")
    print(chunk.page_content[:1000])
    if len(chunk.page_content) > 1000:
        print("……（预览截断，完整内容保存在 semantic_docs 中）")


网页文档数：1
原文总字符数：5960
语义文本块数：4

文本块 1 | 字符数：0
元数据：{'source': 'https://docs.langchain.com/langsmith/deploy-overview', 'title': 'Deploy agents - Docs by LangChain', 'description': 'Deploy agents to production on LangSmith Agent Server, on LangSmith Cloud, BYOC, your own infrastructure, or a standalone server.', 'language': 'en', 'chunk_index': 1}


文本块 2 | 字符数：33
元数据：{'source': 'https://docs.langchain.com/langsmith/deploy-overview', 'title': 'Deploy agents - Docs by LangChain', 'description': 'Deploy agents to production on LangSmith Agent Server, on LangSmith Cloud, BYOC, your own infrastructure, or a standalone server.', 'language': 'en', 'chunk_index': 2}
Deploy agents - Docs by LangChain

文本块 3 | 字符数：5904
元数据：{'source': 'https://docs.langchain.com/langsmith/deploy-overview', 'title': 'Deploy agents - Docs by LangChain', 'description': 'Deploy agents to production on LangSmith Agent Server, on LangSmith Cloud, BYOC, your own infrastructure, or a standalone server.', 'language': 'en', 'c